# 01 — Gaussian Process Surrogate (from scratch)
**Model:** Gaussian Process Regression — numpy only, no sklearn GP modules.

**Why GP for this dataset:**
Only 17 training points. GP gives exact interpolation at training points
plus calibrated uncertainty bands everywhere else.
Uncertainty comes from the mathematics of the model — not from sampling.

**Notebook role in pipeline:**
```
data_utils.py  →  01_gp_surrogate.ipynb  →  04_model_comparison.ipynb
                                          →  ../analysis/abaqus_material_cards.inp
```

**Outputs saved to** `../analysis/`:
- `tension_compression_asymmetry.png`
- `gp_predictions.png`
- `gp_loo_residuals.png`
- `gp_derived_quantities.png`
- `gp_results.pkl`  ← loaded by 04_model_comparison.ipynb


---
## Cell 1 — Imports

Only numpy, scipy.optimize (for hyperparameter fitting), and plotting.
No sklearn GP modules anywhere.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, sys, os
from scipy.optimize import minimize   # used only for log-marginal-likelihood optimisation
sys.path.insert(0, os.path.dirname(os.path.abspath('data_utils.py')))
from data_utils import load_data, plot_loo_residuals, load_asymmetry, load_all_dat, TARGETS, PALETTE
from sklearn.metrics import mean_absolute_error, r2_score  # metrics only, not GP
from sklearn.model_selection import LeaveOneOut             # CV splitter only
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
print('Imports OK — no sklearn GP used')

---
## Cell 2 — Load Data

In [ ]:
# ── EDIT PATH ────────────────────────────────────────────────────────────
DATA_PATH = '../analysis/elastic_constants_fecr.csv'
# ─────────────────────────────────────────────────────────────────────────
d = load_data(DATA_PATH)
df, X, X_pred, x_pred_atoms = d['df'], d['X'], d['X_pred'], d['x_pred_atoms']
targets, alpha_noise, flagged = d['targets'], d['alpha'], d['flagged']
# alpha_noise[i] = noise VARIANCE for point i  (GPa²)
# flagged points: 10² = 100 GPa²  |  normal: 1² = 1 GPa²

---
## Cell 3 — Pre-ML: Tension-Compression Asymmetry

Checks whether DFT stress responds symmetrically to +ε and −ε.

**Harmonic solid:** P(+ε) + P(−ε) = 2·P(0) → asymmetry index = 0

Set `DAT_ROOT` to the folder containing `fe16cr00/`, `fe15cr01/`, ... subdirs.
If .dat files are not accessible locally, skip — does not affect ML training.

In [ ]:
DAT_ROOT = '../'   # ← EDIT: root of your DFT data directory
asym_df = load_asymmetry(df, DAT_ROOT)
if len(asym_df) == 0:
    print('No .dat files found — skipping asymmetry analysis.')
else:
    print(asym_df[['tag','n_cr','P_plus','P_minus','asymmetry_kbar']].to_string())

In [ ]:
if len(asym_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ax = axes[0]
    ax.plot(asym_df['n_cr'], asym_df['P_plus'],  'o-', color='firebrick', label='P(+ε) tension')
    ax.plot(asym_df['n_cr'], asym_df['P_minus'], 's-', color='steelblue', label='P(−ε) compression')
    if asym_df['P_zero'].notna().any():
        ax.plot(asym_df['n_cr'], asym_df['P_zero'], '^--', color='gray', alpha=0.6, label='P(0) ref')
    fl = asym_df[asym_df['loose_thr']]
    ax.scatter(fl['n_cr'], fl['P_plus'],  marker='x', s=140, color='firebrick', zorder=5)
    ax.scatter(fl['n_cr'], fl['P_minus'], marker='x', s=140, color='steelblue', zorder=5)
    ax.set_xlabel('Cr atoms'); ax.set_ylabel('Pressure (kbar)')
    ax.set_title('Hydrostatic Pressure: Tension vs Compression')
    ax.legend(); ax.grid(alpha=0.3)
    ax2 = axes[1]
    cols = ['tomato' if f else 'steelblue' for f in asym_df['loose_thr']]
    ax2.bar(asym_df['n_cr'], asym_df['asymmetry_kbar'], color=cols, edgecolor='k', lw=0.5)
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Cr atoms'); ax2.set_ylabel('Asymmetry index (kbar)')
    ax2.set_title('Anharmonicity: P(+ε)+P(−ε)−2P(0)')
    ax2.text(0.97, 0.95, '✕ = loose_thr', transform=ax2.transAxes,
             ha='right', va='top', color='tomato', fontsize=10)
    ax2.grid(alpha=0.3, axis='y')
    plt.suptitle('Pre-ML: Anharmonicity Check', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../analysis/tension_compression_asymmetry.png', bbox_inches='tight')
    plt.show()
    print('Saved. Bars near zero → linear elastic assumption is valid.')

---
## Cell 4 — GP Implementation from Scratch

### The mathematics — step by step

A Gaussian Process defines a **prior distribution over functions**.
We specify it with a mean function (zero here) and a **kernel** k(x, x')
that encodes how similar two inputs x and x' are.

#### Kernel functions
Given inputs x, x' (here: Cr fractions), the kernel returns a scalar
measuring their 'similarity'. Two kernels are implemented:

**RBF (Radial Basis Function / squared exponential):**
```
k(x,x') = σ² · exp( −||x−x'||² / (2·ℓ²) )
```
σ² = signal variance (amplitude), ℓ = length-scale (how fast similarity decays).
Implies infinitely smooth functions. Good for smooth Cij trends.

**Matérn-5/2:**
```
k(x,x') = σ² · (1 + √5·r/ℓ + 5r²/(3ℓ²)) · exp(−√5·r/ℓ)
where r = |x−x'|
```
Implies twice-differentiable (but not infinitely smooth) functions.
More realistic for physical alloy systems with composition-dependent anomalies.

#### Training: maximise the log marginal likelihood
Given training data (X, y) and noise variances σ²_i per point:
```
K_noisy = K(X,X) + diag(σ²_noise)     # covariance matrix + noise
log p(y|X,θ) = −½ yᵀ K_noisy⁻¹ y − ½ log|K_noisy| − n/2 log(2π)
```
We optimise θ = (σ², ℓ) to maximise this. This is the **only** use of
scipy.optimize — everything else is pure numpy.

#### Prediction
For a new point x*:
```
μ(x*) = k(x*,X) · K_noisy⁻¹ · y              # posterior mean
σ²(x*) = k(x*,x*) − k(x*,X) · K_noisy⁻¹ · k(X,x*)  # posterior variance
```
σ(x*) is the **standard deviation** — the uncertainty band in the plots.
It is large where training data is sparse, zero at training points (noise-free limit).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# KERNEL FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────

def kernel_rbf(X1, X2, sigma2, ell):
    """
    RBF (squared exponential) kernel.
    k(x,x') = sigma2 * exp( -||x-x'||^2 / (2*ell^2) )

    Parameters
    ----------
    X1, X2 : (n,1), (m,1) arrays
    sigma2  : float — signal variance  (amplitude squared)
    ell     : float — length-scale

    Returns
    -------
    K : (n, m) kernel matrix
    """
    # Squared distances: ||x_i - x_j||^2
    # Broadcasting: (n,1,d) - (1,m,d) → (n,m,d) → sum over d
    diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]   # (n,m,1)
    r2   = np.sum(diff**2, axis=-1)                        # (n,m)
    return sigma2 * np.exp(-r2 / (2.0 * ell**2))


def kernel_matern52(X1, X2, sigma2, ell):
    """
    Matérn-5/2 kernel.
    k(x,x') = sigma2 * (1 + sqrt(5)*r/ell + 5*r^2/(3*ell^2)) * exp(-sqrt(5)*r/ell)
    where r = ||x - x'||

    Parameters
    ----------
    X1, X2 : (n,1), (m,1) arrays
    sigma2  : float — signal variance
    ell     : float — length-scale

    Returns
    -------
    K : (n, m) kernel matrix
    """
    diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]
    r2   = np.sum(diff**2, axis=-1)
    r    = np.sqrt(np.maximum(r2, 0.0))   # clip tiny negatives from float errors
    s5   = np.sqrt(5.0) * r / ell
    return sigma2 * (1.0 + s5 + s5**2 / 3.0) * np.exp(-s5)


KERNELS = {
    'RBF':        kernel_rbf,
    'Matern-5/2': kernel_matern52
}
print('Kernel functions defined: RBF, Matern-5/2')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# GAUSSIAN PROCESS REGRESSOR (numpy)
# ─────────────────────────────────────────────────────────────────────────

class GaussianProcessScratch:
    """
    GP Regression from scratch — numpy + scipy.optimize only.

    Hyperparameters: sigma2 (signal variance), ell (length-scale).
    Optimised by maximising the log marginal likelihood.

    Parameters
    ----------
    kernel     : str  — 'RBF' or 'Matern-5/2'
    n_restarts : int  — number of random restarts for optimisation
    seed       : int
    """

    def __init__(self, kernel='Matern-5/2', n_restarts=10, seed=42):
        self.kernel_name = kernel
        self.kernel_fn   = KERNELS[kernel]
        self.n_restarts  = n_restarts
        self.seed        = seed
        # Fitted attributes (set by .fit)
        self.sigma2      = None   # optimised signal variance
        self.ell         = None   # optimised length-scale
        self.log_ml      = None   # log marginal likelihood at optimum
        self.alpha_vec   = None   # K_noisy^{-1} y  (precomputed for prediction)
        self.L           = None   # Cholesky factor of K_noisy
        self.X_train     = None
        self.noise_var   = None   # per-point noise variances

    # ── Internal: build noisy covariance and compute log marginal likelihood ──
    def _log_marginal_likelihood(self, log_params, X, y, noise_var):
        """
        Negative log marginal likelihood (minimised by scipy).

        We optimise in log-space so parameters stay positive:
            log_params = [log(sigma2), log(ell)]

        Log marginal likelihood:
            log p(y|X,θ) = -½ yᵀ K⁻¹ y - ½ log|K| - n/2 log(2π)

        Computed via Cholesky for numerical stability:
            K = L Lᵀ  →  log|K| = 2 sum(log diag(L))
            K⁻¹ y  solved via forward/back substitution
        """
        sigma2 = np.exp(log_params[0])
        ell    = np.exp(log_params[1])
        n      = len(y)
        K      = self.kernel_fn(X, X, sigma2, ell)
        K_noisy = K + np.diag(noise_var)
        # Add jitter for numerical stability (avoids near-singular K)
        K_noisy += 1e-8 * np.eye(n)
        try:
            L = np.linalg.cholesky(K_noisy)   # lower triangular: K = L Lᵀ
        except np.linalg.LinAlgError:
            return 1e10  # not positive definite — skip this parameter set
        # Solve L alpha_L = y  (forward substitution)
        alpha_L = np.linalg.solve(L, y)
        # log|K| = 2 * sum(log diag(L))
        log_det = 2.0 * np.sum(np.log(np.diag(L)))
        # log p = -½ ||alpha_L||² - ½ log|K| - n/2 log(2π)
        lml = -0.5 * np.dot(alpha_L, alpha_L) - 0.5 * log_det - 0.5 * n * np.log(2*np.pi)
        return -lml   # return NEGATIVE (scipy minimises)

    # ── Fit: optimise hyperparameters ─────────────────────────────────────────
    def fit(self, X, y, noise_var):
        """
        Optimise sigma2, ell by maximising log marginal likelihood.
        Multiple random restarts to avoid local optima.

        Parameters
        ----------
        X         : (n,1) feature array
        y         : (n,)  target array
        noise_var : (n,)  per-point noise variance array
        """
        self.X_train   = X.copy()
        self.noise_var = noise_var.copy()
        rng = np.random.default_rng(self.seed)

        best_lml, best_params = np.inf, None
        # Bounds for log(sigma2), log(ell) — keep parameters in reasonable range
        bounds = [(-4, 8), (-6, 2)]

        for _ in range(self.n_restarts):
            # Random start in log-space
            log0 = rng.uniform([b[0] for b in bounds],
                                [b[1] for b in bounds])
            res = minimize(self._log_marginal_likelihood,
                           log0,
                           args=(X, y, noise_var),
                           method='L-BFGS-B',
                           bounds=bounds)
            if res.fun < best_lml:
                best_lml    = res.fun
                best_params = res.x

        self.sigma2  = np.exp(best_params[0])
        self.ell     = np.exp(best_params[1])
        self.log_ml  = -best_lml   # store positive LML

        # Precompute K_noisy^{-1} y via Cholesky for fast prediction
        n = len(y)
        K = self.kernel_fn(X, X, self.sigma2, self.ell)
        K_noisy = K + np.diag(noise_var) + 1e-8 * np.eye(n)
        self.L = np.linalg.cholesky(K_noisy)
        # alpha_vec = K_noisy^{-1} y
        # Solve L Lᵀ alpha_vec = y in two triangular steps
        v = np.linalg.solve(self.L, y)              # forward: L v = y
        self.alpha_vec = np.linalg.solve(self.L.T, v)  # backward: Lᵀ α = v
        return self

    # ── Predict: posterior mean and standard deviation ────────────────────────
    def predict(self, X_star, return_std=True):
        """
        Posterior mean and std at new inputs X_star.

        μ(x*) = k(x*,X) · K_noisy⁻¹ · y  =  k_star · alpha_vec
        σ²(x*) = k(x*,x*) - k_star · K_noisy⁻¹ · k_starᵀ
               = k(x*,x*) - ||L⁻¹ k_starᵀ||²

        Parameters
        ----------
        X_star     : (m,1)
        return_std : bool

        Returns
        -------
        mu  : (m,) posterior mean
        std : (m,) posterior std  (only if return_std=True)
        """
        k_star = self.kernel_fn(X_star, self.X_train,
                                self.sigma2, self.ell)  # (m, n)
        mu = k_star @ self.alpha_vec                     # (m,)

        if not return_std:
            return mu

        # Posterior variance at each x*
        # v = L^{-1} k_starᵀ  →  shape (n, m)
        v = np.linalg.solve(self.L, k_star.T)   # forward solve
        k_ss = self.kernel_fn(X_star, X_star,
                               self.sigma2, self.ell)  # (m,m)
        var = np.diag(k_ss) - np.sum(v**2, axis=0)    # (m,)
        var = np.maximum(var, 0.0)   # clip tiny negatives from float errors
        return mu, np.sqrt(var)

print('GaussianProcessScratch class defined')
print('Dependencies: numpy only (scipy.optimize used for hyperparameter search only)')

---
## Cell 5 — Train GPs + LOO-CV

Both kernels trained per target. Best selected by LOO-CV R².

**LOO-CV:** At each fold, one point is held out, the GP is re-fitted
on the remaining 16 points using the **same optimised hyperparameters**
from the full fit, then we predict the held-out point.
This avoids re-running the expensive optimisation 17 times per kernel per target.

In [ ]:
def loo_cv_gp(gp_full, X, y, noise_var):
    """
    LOO-CV using the already-optimised hyperparameters from gp_full.
    Re-fits a new GP at each fold with fixed sigma2, ell.
    """
    loo_preds, loo_true = [], []
    for tr, te in LeaveOneOut().split(X):
        gp_loo = GaussianProcessScratch(
            kernel=gp_full.kernel_name, n_restarts=1, seed=0)
        # Fix hyperparameters — no re-optimisation per fold
        gp_loo.X_train   = X[tr]
        gp_loo.noise_var = noise_var[tr]
        gp_loo.sigma2    = gp_full.sigma2
        gp_loo.ell       = gp_full.ell
        # Recompute Cholesky and alpha for the leave-one-out training set
        n_tr = len(tr)
        K    = gp_loo.kernel_fn(X[tr], X[tr], gp_loo.sigma2, gp_loo.ell)
        K_n  = K + np.diag(noise_var[tr]) + 1e-8 * np.eye(n_tr)
        gp_loo.L         = np.linalg.cholesky(K_n)
        v                = np.linalg.solve(gp_loo.L, y[tr])
        gp_loo.alpha_vec = np.linalg.solve(gp_loo.L.T, v)
        # Predict held-out point
        mu_te = gp_loo.predict(X[te], return_std=False)
        loo_preds.append(float(mu_te[0]))
        loo_true.append(float(y[te[0]]))
    loo_preds = np.array(loo_preds)
    loo_true  = np.array(loo_true)
    return dict(loo_preds=loo_preds, loo_true=loo_true,
                residuals=loo_true - loo_preds,
                mae=mean_absolute_error(loo_true, loo_preds),
                r2=r2_score(loo_true, loo_preds))


gp_store  = {}   # gp_store[target][kernel] = {'gp': ..., 'loo': ...}
best_gps  = {}   # best GP per target (by LOO R²)

for tname in TARGETS:
    y = targets[tname]
    print(f'\n── {tname} ─────────────────────────────────────')
    gp_store[tname] = {}
    best_r2 = -np.inf
    for kname in KERNELS:
        gp = GaussianProcessScratch(kernel=kname, n_restarts=10)
        gp.fit(X, y, noise_var=alpha_noise)
        loo = loo_cv_gp(gp, X, y, alpha_noise)
        print(f'  {kname:12s}  σ²={gp.sigma2:.2f}  ℓ={gp.ell:.4f}  '
              f'logML={gp.log_ml:.2f}  MAE={loo["mae"]:.2f} GPa  R²={loo["r2"]:.4f}')
        gp_store[tname][kname] = {'gp': gp, 'loo': loo}
        if loo['r2'] > best_r2:
            best_r2 = loo['r2']
            best_gps[tname] = gp
            best_kname = kname
    print(f'  → Best kernel: {best_kname} (R²={best_r2:.4f})')

---
## Cell 6 — Prediction Plots with Uncertainty Bands

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, tname in zip(axes, TARGETS):
    gp = best_gps[tname]
    mu, sig = gp.predict(X_pred, return_std=True)
    col = PALETTE[tname]
    ax.fill_between(x_pred_atoms, mu-2*sig, mu+2*sig, alpha=0.12, color=col, label='±2σ')
    ax.fill_between(x_pred_atoms, mu-sig,   mu+sig,   alpha=0.28, color=col, label='±1σ')
    ax.plot(x_pred_atoms, mu, '-', color=col, lw=2, label='GP mean')
    ax.scatter(df['n_cr'][~flagged], targets[tname][~flagged],
               color='black', s=50, zorder=5, label='DFT')
    ax.scatter(df['n_cr'][flagged], targets[tname][flagged],
               color='tomato', s=70, marker='D', zorder=5, label='⚠️ flagged')
    ax.set_xlabel('Cr atoms (out of 16)'); ax.set_ylabel(f'{tname} (GPa)')
    ax.set_title(f'{tname} — GP ({gp.kernel_name})')
    ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(-0.5, 16.5)
plt.suptitle('GP Surrogate: Fe-Cr Elastic Constants (from scratch)', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/gp_predictions.png', bbox_inches='tight')
plt.show(); print('Saved: gp_predictions.png')

---
## Cell 7 — LOO-CV Residuals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tname in zip(axes, TARGETS):
    gp  = best_gps[tname]
    loo = gp_store[tname][gp.kernel_name]['loo']
    plot_loo_residuals(ax, loo['residuals'], df['n_cr'].values, flagged,
                       f"{tname} LOO-CV ({gp.kernel_name})\n"
                       f"MAE={loo['mae']:.2f} GPa | R²={loo['r2']:.3f}")
plt.suptitle('GP LOO-CV Residuals  (red = flagged)', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/gp_loo_residuals.png', bbox_inches='tight')
plt.show(); print('Saved: gp_loo_residuals.png')

---
## Cell 8 — Derived Quantities: B, Zener A, E[100], E[111]

| Quantity | Formula |
|---|---|
| Bulk modulus B | (C11+2·C12)/3 |
| Zener anisotropy A | 2·C44/(C11−C12) |
| Young's E[100] | (C11−C12)·(C11+2·C12)/(C11+C12) |
| Young's E[111] | 3·C44·(C11+2·C12)/(C11+2·C12+C44) |

Uncertainty propagated analytically (first-order) for B and Zener A.

⚠ E[111] formula: verify against Nye *Physical Properties of Crystals* before citing.

In [ ]:
c11, s11 = best_gps['C11'].predict(X_pred, return_std=True)
c12, s12 = best_gps['C12'].predict(X_pred, return_std=True)
c44, s44 = best_gps['C44'].predict(X_pred, return_std=True)

B      = (c11 + 2*c12) / 3.0
B_std  = np.sqrt(s11**2 + 4*s12**2) / 3.0
denom  = c11 - c12
ZA     = 2*c44 / denom
ZA_std = np.sqrt((2/denom * s44)**2 +
                  (2*c44/denom**2 * s11)**2 +
                  (2*c44/denom**2 * s12)**2)
E100   = denom * (c11 + 2*c12) / (c11 + c12)
E111   = 3*c44*(c11 + 2*c12) / (c11 + 2*c12 + c44)

print(f'B      : {B.min():.1f} – {B.max():.1f} GPa')
print(f'ZenerA : {ZA.min():.3f} – {ZA.max():.3f}')
print(f'E[100] : {E100.min():.1f} – {E100.max():.1f} GPa')
print(f'E[111] : {E111.min():.1f} – {E111.max():.1f} GPa')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
pdata = [
    (axes[0,0], B,    B_std,  'Bulk Modulus B (GPa)',       'royalblue',  None),
    (axes[0,1], ZA,   ZA_std, 'Zener Anisotropy A',         'darkorchid', 1.0),
    (axes[1,0], E100, None,   "Young's Modulus E[100] (GPa)",'darkorange', None),
    (axes[1,1], E111, None,   "Young's Modulus E[111] (GPa)",'seagreen',   None),
]
for ax, ym, ys, title, col, hl in pdata:
    ax.plot(x_pred_atoms, ym, '-', color=col, lw=2)
    if ys is not None:
        ax.fill_between(x_pred_atoms, ym-ys, ym+ys, alpha=0.25, color=col, label='±1σ')
    if hl:
        ax.axhline(hl, color='gray', ls='--', lw=1, label=f'isotropic A={hl}')
    if 'Bulk' in title and 'B' in df.columns:
        ax.scatter(df['n_cr'], df['B'], color='black', s=50, zorder=5, label='DFT')
        ax.scatter(df.loc[flagged,'n_cr'], df.loc[flagged,'B'],
                   color='tomato', s=70, marker='D', zorder=6, label='flagged')
    ax.set_xlabel('Cr atoms (out of 16)'); ax.set_title(title)
    ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(-0.5, 16.5)
plt.suptitle('GP: Derived Elastic Quantities', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/gp_derived_quantities.png', bbox_inches='tight')
plt.show(); print('Saved: gp_derived_quantities.png')

---
## Cell 9 — Export Results for Model Comparison

In [ ]:
export = {}
for tname in TARGETS:
    gp  = best_gps[tname]
    loo = gp_store[tname][gp.kernel_name]['loo']
    mu, sig = gp.predict(X_pred, return_std=True)
    export[tname] = {
        'mu':          mu,
        'std':         sig,
        'mae':         loo['mae'],
        'r2':          loo['r2'],
        'loo_true':    loo['loo_true'],
        'loo_preds':   loo['loo_preds'],
        'residuals':   loo['residuals'],
        'best_kernel': gp.kernel_name,
        'sigma2':      gp.sigma2,
        'ell':         gp.ell,
        'log_ml':      gp.log_ml
    }
with open('../analysis/gp_results.pkl', 'wb') as f:
    pickle.dump(export, f)
print('Saved: gp_results.pkl')
print('\nGP Summary:')
for t, v in export.items():
    print(f'  {t}: kernel={v["best_kernel"]:12s}  '
          f'σ²={v["sigma2"]:.2f}  ℓ={v["ell"]:.4f}  '
          f'logML={v["log_ml"]:.2f}  MAE={v["mae"]:.2f} GPa  R²={v["r2"]:.4f}')

---
## Cell 10 — ABAQUS Material Card Export

Queries GP at selected Cr compositions → writes `*Elastic, type=ANISOTROPIC` snippets.

**Units:** GPa here. ABAQUS typically expects MPa — multiply Cij × 1000.

**Cubic Voigt order:**
D1111, D1122, D2222, D1133, D2233, D3333, D1212, D1313 (line 1) + D2323 (line 2)
= C11, C12, C11, C12, C12, C11, C44, C44 (line 1) + C44 (line 2)

⚠ Verify `*Elastic ANISOTROPIC` keyword format against your ABAQUS version docs.

In [ ]:
FEM_CR_ATOMS = [0, 4, 8, 12, 16]   # ← edit as needed

rows, lines = [], []
lines += ['** Fe-Cr ABAQUS Material Cards — GP Surrogate (from scratch)',
          '** Units: GPa  (multiply by 1000 for MPa)',
          '** Verify *Elastic ANISOTROPIC format for your ABAQUS version', '']

for ncr in FEM_CR_ATOMS:
    xv = np.array([[ncr / 16.0]])
    c11v, s11v = best_gps['C11'].predict(xv)
    c12v, s12v = best_gps['C12'].predict(xv)
    c44v, s44v = best_gps['C44'].predict(xv)
    c11v, c12v, c44v = float(c11v), float(c12v), float(c44v)
    s11v, s12v, s44v = float(s11v), float(s12v), float(s44v)
    bv  = (c11v + 2*c12v) / 3
    zav = 2*c44v / (c11v - c12v)
    label = f'Fe{16-ncr}Cr{ncr}'
    rows.append({'n_cr': ncr, 'x_cr': ncr/16,
                 'C11_GPa': round(c11v,2), 'C11_std': round(s11v,2),
                 'C12_GPa': round(c12v,2), 'C12_std': round(s12v,2),
                 'C44_GPa': round(c44v,2), 'C44_std': round(s44v,2),
                 'B_GPa':   round(bv,2),   'ZenerA':  round(zav,4)})
    lines += [
        f'** {label}  B={bv:.2f} GPa  ZenerA={zav:.4f}  '
        f'C11±{s11v:.2f}  C12±{s12v:.2f}  C44±{s44v:.2f}',
        f'*Material, name={label}',
        '*Elastic, type=ANISOTROPIC',
        f'{c11v:.3f}, {c12v:.3f}, {c11v:.3f}, {c12v:.3f}, '
        f'{c12v:.3f}, {c11v:.3f}, {c44v:.3f}, {c44v:.3f},',
        f'{c44v:.3f}', '']

fem_df = pd.DataFrame(rows)
fem_df.to_csv('../analysis/fem_material_inputs_gp.csv', index=False)
txt = '\n'.join(lines)
with open('../analysis/abaqus_material_cards.inp', 'w') as f:
    f.write(txt)
print(txt)
print('Saved: fem_material_inputs_gp.csv  |  abaqus_material_cards.inp')

---
## Cell 11 — Summary

**Open caveats (always carry these forward):**
1. C11−C12 denominator `3ε` — taken as likely correct; verify vs primary reference before citing
2. E[111] formula — verify vs Nye *Physical Properties of Crystals*
3. fe02cr14/15/16 — mixed magnetic setup, down-weighted not removed
4. ABAQUS card format — verify `*Elastic ANISOTROPIC` for your version

In [ ]:
print('='*60)
print('GP SURROGATE — SUMMARY (from scratch)')
print('='*60)
print(f'Points: {len(df)}  |  Flagged: {flagged.sum()}')
print(f'Kernels: {list(KERNELS.keys())}')
print(f'Hyperparameter optimisation: L-BFGS-B, {best_gps["C11"].n_restarts} random restarts')
print()
print('Best model per target (LOO-CV):')
for t, v in export.items():
    print(f'  {t}: {v["best_kernel"]:12s}  '
          f'σ²={v["sigma2"]:.2f}  ℓ={v["ell"]:.4f}  '
          f'MAE={v["mae"]:.2f} GPa  R²={v["r2"]:.4f}')
print()
print('Outputs:')
for fn in ['tension_compression_asymmetry.png', 'gp_predictions.png',
           'gp_loo_residuals.png', 'gp_derived_quantities.png',
           'gp_results.pkl', 'fem_material_inputs_gp.csv',
           'abaqus_material_cards.inp']:
    print(f'  ../analysis/{fn}')